# $\color{teal}{\text{Shipping a Change}}$

At this point, Need to Know has a workflow that tests and publishes an image for every commit, a release for every version, a script that deploys and rolls back, cache headers, and a small database of usage counts that changed over four releases. In the lecture, we read all of it. In this notebook, we'll run the parts we can run on our own machine: read the workflow as data, check every cache header, and take the usage database through its migrations, including the step that can't be undone.

Run it from the repository on branch `example-l8`, after `uv sync --extra dev`. Nothing here needs Docker or the instance.

In [ ]:
import json
import logging
import sqlite3
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.disable(logging.INFO)          # keep the app's request log out of the way

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
print(sorted(p.name for p in (ROOT / ".github" / "workflows").iterdir()))

"""
['ci.yml', 'release.yml']
"""

## $\color{teal}{\text{What does our workflow actually say?}}$

A workflow is YAML, so we can load it like any other data and ask it questions. Here are its triggers, and for each job, what it waits for and what its token may do.

In [ ]:
import yaml

ci = yaml.safe_load((ROOT / ".github" / "workflows" / "ci.yml").read_text())
print("triggers:", list(ci[True]))          # YAML reads the key `on` as the boolean True
print("default permissions:", ci["permissions"])
for name, job in ci["jobs"].items():
    print(f"{name:8} needs={job.get('needs', '-')!s:5} if={job.get('if', '-')}  permissions={job.get('permissions', 'default')}")

"""
triggers: ['push', 'pull_request', 'workflow_dispatch']
default permissions: {'contents': 'read'}
test     needs=-     if=-  permissions=default
publish  needs=test  if=github.event_name == 'push'  permissions={'contents': 'read', 'packages': 'write'}
"""

**Q. Only one job can push to ghcr.io. Which line of the output tells us, and what would happen if we deleted it?**

<details>
<summary>Click here for an answer</summary>


The `publish` job's `permissions={'contents': 'read', 'packages': 'write'}`. Without it, the job gets the default from the top of the file, `contents: read`, and `docker push` is refused with a permission error. The tests would still pass; we'd just stop publishing images.

</details>

In [ ]:
test = ci["jobs"]["test"]
print("matrix:", test["strategy"]["matrix"])
print("concurrency:", ci["concurrency"])
for step in test["steps"]:
    print(" ", step.get("uses") or step.get("run"))

"""
matrix: {'python': ['3.12', '3.13']}
concurrency: {'group': '${{ github.workflow }}-${{ github.ref }}', 'cancel-in-progress': "${{ github.event_name == 'pull_request' }}"}
  actions/checkout@v4
  astral-sh/setup-uv@v6
  uv sync --frozen --extra dev
  uv run ruff check src tests
  uv run pytest -q
"""

**Q. Our concurrency rule cancels older runs on pull requests but not on `main`. Where in the output is that decided?**

<details>
<summary>Click here for an answer</summary>


In `cancel-in-progress: ${{ github.event_name == 'pull_request' }}`. GitHub evaluates the expression for each run: it is `true` for a pull request and `false` for a push, so a push to `main` always finishes, and its image is never half-published.

</details>

## $\color{teal}{\text{Which response may be kept, and by whom?}}$

Every response gets a **Cache-Control** header from one middleware in `api/main.py`. FastAPI's `TestClient` calls the app in-process, so we can check every rule without starting a server.

In [ ]:
from fastapi.testclient import TestClient

from needtoknow.api.main import app
from needtoknow.config import settings

lease = (ROOT / "scripts" / "smoke-lease.pdf").read_bytes()
with TestClient(app) as client:     # `with` runs the app's startup, which loads the detector
    for label, r in [("GET /", client.get("/")),
                     (f"GET /static/app.js?v={settings.commit}", client.get(f"/static/app.js?v={settings.commit}")),
                     ("GET /static/app.js?v=bed9ddd", client.get("/static/app.js?v=bed9ddd")),
                     ("POST /api/detect", client.post("/api/detect", files={"file": ("lease.pdf", lease, "application/pdf")})),
                     ("GET /api/stats", client.get("/api/stats"))]:
        print(f"{label:30} {r.status_code}  {r.headers.get('cache-control')}")

"""
GET /                          200  no-cache
GET /static/app.js?v=dev       200  public, max-age=31536000, immutable
GET /static/app.js?v=bed9ddd   200  no-cache
POST /api/detect               200  no-store
GET /api/stats                 200  public, max-age=60
"""

Running from source, the commit is `dev`, so the current script is `app.js?v=dev`. In the image, it's the seven characters CI passed in.

**Q. `/api/stats` says `public, max-age=60` here, but returns `{"counting": false}`. Why is counting off?**

<details>
<summary>Click here for an answer</summary>


Because `NTK_DB_PATH` is empty when we run from source: the Dockerfile sets it to `/data/needtoknow.db` in the image, and nothing sets it here. We'll turn it on with a temporary database in the next section.

</details>

In [ ]:
import hashlib

with TestClient(app) as client:
    page = client.get("/")
    again = client.get("/", headers={"If-None-Match": page.headers["etag"]})
print("ETag:", page.headers["etag"], " bytes:", len(page.content))
print("our own hash of the page:", '"' + hashlib.sha256(page.content).hexdigest()[:16] + '"')
print("asking again with that ETag:", again.status_code, " bytes:", len(again.content))

"""
ETag: "c02e3705180e8ccf"  bytes: 2986
our own hash of the page: "c02e3705180e8ccf"
asking again with that ETag: 304  bytes: 0
"""

**Q. A browser has `/` from yesterday's deploy and asks with yesterday's ETag. What does it get?**

<details>
<summary>Click here for an answer</summary>


The whole page, with a `200` and a new ETag. The ETag is a hash of the page, and the page includes the commit in its script address, so every deploy changes it. That's exactly what we want: `no-cache` plus an ETag means an unchanged page costs a few bytes, and a changed one is never served stale.

</details>

## $\color{teal}{\text{How does a database change safely?}}$

Now the usage counts. We'll point the app at a **temporary database**, apply the first migration only (that's v0.3.0), count a few documents, and then go through the expand and the contract one step at a time. The migrations are the same files the image runs.

In [ ]:
import tempfile

from needtoknow import migrate, store

tmp = Path(tempfile.mkdtemp())
db = str(tmp / "needtoknow.db")
everything = migrate.available()
print([name for _, name, _ in everything])

migrate.available = lambda: everything[:1]          # pretend the image is v0.3.0
migrate.up(db)
con = sqlite3.connect(db)
with con:
    con.execute("INSERT INTO usage VALUES ('2026-10-14', 7, 30), ('2026-10-15', 2, 9)")   # what v0.3.0 counted
print(con.execute("SELECT * FROM usage").fetchall())

"""
['0001_usage.sql', '0002_usage_by_detector.sql', '0003_drop_usage.sql']
applied 0001_usage.sql
schema is at 1
[('2026-10-14', 7, 30), ('2026-10-15', 2, 9)]
"""

In [ ]:
migrate.available = lambda: everything[:2]          # now v0.4.0: the expand step
migrate.up(db)
print(con.execute("SELECT * FROM usage_by_detector").fetchall())
print("old total:", con.execute("SELECT SUM(documents) FROM usage").fetchone()[0],
      " new total:", con.execute("SELECT SUM(documents) FROM usage_by_detector").fetchone()[0])
print(sorted(p.name for p in (tmp / "backups").iterdir()))

"""
backed up to /tmp/tmp0uquvl6r/backups/needtoknow-20261015T214816Z-before-0002.db
applied 0002_usage_by_detector.sql
schema is at 2
[('2026-10-14', 'regex', 7, 30), ('2026-10-15', 'regex', 2, 9)]
old total: 9  new total: 9
['needtoknow-20261015T214816Z-before-0002.db']
"""

**Q. Why does the backup's name say `before-0002`, and when would we need it?**

<details>
<summary>Click here for an answer</summary>


It is the database exactly as it was before migration 0002 ran, so it matches the schema v0.3.0 expects. We'd need it only if 0002 went wrong in a way a rollback can't fix. After an expand step that rarely happens, since v0.3.0 still finds its table and keeps working.

</details>

The check we just printed is worth running for real before any contract step: if the totals in the old and new tables disagree, something isn't writing both.

In [ ]:
migrate.available = lambda: everything                # v0.6.0: the contract step
migrate.up(db)
migrate.status(db)
print(sorted(p.name for p in (tmp / "backups").iterdir()))

"""
backed up to /tmp/tmp0uquvl6r/backups/needtoknow-20261015T214816Z-before-0003.db
applied 0003_drop_usage.sql
schema is at 3
applied  0001_usage.sql
applied  0002_usage_by_detector.sql
applied  0003_drop_usage.sql
schema is at 3
['needtoknow-20261015T214816Z-before-0002.db', 'needtoknow-20261015T214816Z-before-0003.db']
"""

Now suppose someone rolls the image back to v0.4.0. Its code still writes the old table. Here is that write, by hand:

In [ ]:
try:
    with con:
        con.execute("INSERT INTO usage (day, documents, items) VALUES ('2026-10-16', 1, 6)")
except sqlite3.OperationalError as exc:
    print("v0.4.0 would fail with:", exc)

"""
v0.4.0 would fail with: no such table: usage
"""

**Q. v0.4.0's `/ready` still says ready on this database. Why doesn't it notice, and what does?**

<details>
<summary>Click here for an answer</summary>


`/ready` checks that the schema is at least the version the code needs: v0.4.0 needs 2 and finds 3, which is allowed, because old code runs on a newer schema during every deploy. The failure only happens when it writes, and `detect` logs it without failing the user's request. `scripts/smoke.sh` notices, because it checks that the document it sent was counted.

</details>

In [ ]:
backup = sorted((tmp / "backups").glob("*before-0003.db"))[-1]
con.close()
Path(db).write_bytes(backup.read_bytes())               # the restore: the app would be stopped first
con = sqlite3.connect(db)
print("schema:", store.schema_version(db), " tables:",
      [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")])

"""
schema: 2  tables: ['schema_migrations', 'usage', 'usage_by_detector']
"""

**Q. We restored the backup. What did we lose?**

<details>
<summary>Click here for an answer</summary>


Every count recorded after the backup was taken, that is, after v0.6.0 was deployed. That's the price of undoing a contract step, and it's why the runbook marks it as the step that can't be undone and the changelog warns before it.

</details>

## $\color{teal}{\text{Take-home message}}$

In this notebook, we read our workflow as data and found the one job allowed to push, checked every cache header without a server, and took a real database through its expand and contract migrations, including a rollback that fails and a restore that loses data. The same files run in the image on the instance, through `scripts/deploy.sh`.

**You've shipped a change, and a change to stored data, the way a team can repeat it!**